# Event Prediction : prédiction de la bactériémie

## Contexte

Les prédicteurs des bactériémies restent mal caractérisés.
L'objectif du défi 1 est de **développer et valider un modèle de prédiction de la survenue d'une bactériémie** chez les patients.

## Données

- **Fichier** : `bacteriemie.csv.gz`
- **Taille** : 11 753 observations provenant de différents patients présentant une suspicion clinique de bactériémie, pour lesquels une analyse de culture de sang a été réalisée à l'Hôpital général de Vienne (Autriche), entre janvier 2006 et décembre 2010.
- **Variables** : le résultat de l'analyse de culture de sang pour la bactériémie et les valeurs de **51 prédicteurs potentiels**.
- **Confidentialité** : afin de protéger la confidentialité des données, cette version a été légèrement modifiée par rapport à l'originale ; la version modifiée a été approuvée par l'Université médicale de Vienne pour un usage public (DC 2019-0054).
- **Dictionnaire des variables** : fichier `cahierVariables.csv`

## Travail demandé

* Développer un modèle permettant de prédire la survenue d'une bactériémie pour un patient.
* Le rendu est un code qui peut fonctionner sur un fichier de même structure que le fichier de données initial et qui retourne la présence (1) ou l'absence (0) de bactériémie pour chaque sujet.

Le script sera utilisé pour prédire un jeu de test qui n'est pas encore mis à disposition. Les performances de l'algorithme seront évaluées. Un bonus sera attribué au groupe qui obtiendra la meilleure performance (**F1 score**, puis **précision** en cas d'égalité) sur le jeu de test.

Les groupes sont définis dans le fichier `GroupesIDS2026.xlsx`.

## Échéances

| Date | Rendu |
|---|---|
| **05/10/2026 à 18h** | [Retour des scripts](https://moodle.unistra.fr/mod/assign/view.php?id=459529) sur Moodle (remise de devoirs : fichier `.r`, `.py` ou `.ipynb`) |
| **07/10/2026 à 18h** | Fichier de prédiction du jeu de test au format `.csv` |
| **08/10/2026, 14h-16h** | Présentations orales et dévoilement du classement |

**Format du fichier de prédiction** : le nom comporte le groupe et un nom (par exemple `Groupe2_godet.csv`). Le fichier contient 2 colonnes :
- `ID` : identifiant issu du fichier de test
- `pred` : uniquement des 0 (absence de bactériémie) ou des 1 (présence)

## Mots-clés

**FR** : Santé, analyse exploratoire des données, nettoyage des données, classification binaire, réduction de la dimensionnalité, sous- et sur-échantillonnage, données déséquilibrées, super-learner, apprentissage profond.

**EN** : Health, Exploratory Data Analysis, Data Cleaning, Categorical Data, Binary classification, Dimensionality reduction, under-/over-sampling, unbalanced weighting, super-learner, deep learning

### 1. EDA 

In [1]:
import polars as pl 

In [2]:
df = pl.read_csv("../data/bacteriemie.csv.gz", null_values="NA")
df

ID,SEX,AGE,MCV,HGB,HCT,PLT,MCH,MCHC,RDW,MPV,LYM,MONO,EOS,BASO,NT,APTT,FIB,SODIUM,POTASS,CA,PHOS,MG,CREA,BUN,HS,GBIL,TP,ALB,AMY,PAMY,LIP,CHE,AP,ASAT,ALAT,GGT,LDH,CK,GLU,TRIG,CHOL,CRP,BASOR,EOSR,LYMR,MONOR,NEU,NEUR,PDW,RBC,WBC,BloodCulture
i64,i64,i64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
1,2,62,99.3,11.5,35.9,307,31.5,31.8,19.5,10.8,0.4,1.7,0.0,0.1,86,28.8,578,137,3.88,2.29,1.2,0.66,0.65,5.7,5.3,0.59,67.0,36.7,30,16,10,5.12,85,22,14,48,284,23,107,105,175,3.94,0.413223,0.0,1.652893,7.024793,22.0,90.909091,10.6,3.7,24.1,"""no"""
3,1,72,85.1,10.3,34.7,182,26.0,30.6,15.0,9.7,0.4,0.2,0.1,0.0,90,29.8,null,141,null,2.21,0.58,null,0.76,19.9,null,0.48,65.3,37.4,null,null,null,5.61,80,28,25,61,null,36,84,null,null,1.42,0.0,0.826446,3.305785,1.652893,11.4,94.214876,11.4,3.9,12.17,"""no"""
5,1,46,96.3,7.4,22.8,64,31.2,32.4,19.7,11.1,1.5,1.2,0.1,0.1,58,36.3,313,147,4.61,1.92,1.51,1.03,1.25,50.6,null,8.42,40.5,22.1,146,null,89,2.52,119,124,135,134,696,40,107,null,null,12.09,0.568182,0.568182,8.522727,6.818182,14.7,83.522727,14.1,2.5,17.45,"""no"""
9,2,38,85.1,13.7,38.7,183,30.2,35.3,12.6,10.0,0.8,0.4,0.0,0.0,95,33.1,490,137,null,2.34,0.97,0.74,0.65,8.5,3.0,0.42,78.4,43.8,84,50,50,6.91,108,35,22,72,null,79,93,152,167,11.17,0.0,0.0,8.333333,4.166667,8.4,87.5,12.2,4.4,9.86,"""no"""
10,1,68,104.5,15.7,46.9,144,34.8,33.5,13.9,10.9,2.2,0.9,0.1,0.0,61,41.8,400,141,4.41,2.08,0.99,0.56,0.82,15.3,5.5,2.4,57.5,30.1,95,57,25,6.79,68,32,11,68,263,75,89,85,144,5.89,0.0,1.0,22.0,9.0,6.8,68.0,12.9,4.3,9.94,"""no"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
62432,1,37,82.0,9.0,25.9,145,28.6,34.6,16.4,9.9,0.0,0.3,0.0,0.0,44,44.1,457,138,5.59,2.17,2.76,1.03,4.1,63.1,5.8,3.06,58.0,32.1,535,14,null,2.56,87,62,28,102,346,150,113,70,89,15.12,0.0,0.0,0.0,1.477833,20.0,98.522167,11.6,3.4,20.34,"""no"""
62436,2,44,97.4,7.9,22.5,248,33.6,34.4,13.3,9.8,1.3,0.4,0.1,0.0,null,null,null,134,3.98,2.2,0.91,0.62,0.56,5.7,8.1,0.7,53.8,26.9,46,34,146,2.6,51,28,22,144,564,61,76,298,null,26.8,0.0,2.083333,27.083333,8.333333,3.0,62.5,11.0,2.4,4.89,"""no"""
62438,1,23,67.4,11.5,38.2,58,20.1,29.8,18.4,null,0.6,0.5,0.0,0.0,null,null,null,137,null,2.24,0.97,null,0.64,12.6,null,null,66.3,42.8,31,null,14,null,155,60,72,36,null,28,85,null,null,4.68,0.0,0.0,10.714286,8.928571,4.5,80.357143,null,5.7,5.62,"""no"""


In [8]:
df.schema

Schema([('ID', Int64),
        ('SEX', Int64),
        ('AGE', Int64),
        ('MCV', Float64),
        ('HGB', Float64),
        ('HCT', Float64),
        ('PLT', Int64),
        ('MCH', Float64),
        ('MCHC', Float64),
        ('RDW', Float64),
        ('MPV', Float64),
        ('LYM', Float64),
        ('MONO', Float64),
        ('EOS', Float64),
        ('BASO', Float64),
        ('NT', Int64),
        ('APTT', Float64),
        ('FIB', Int64),
        ('SODIUM', Int64),
        ('POTASS', Float64),
        ('CA', Float64),
        ('PHOS', Float64),
        ('MG', Float64),
        ('CREA', Float64),
        ('BUN', Float64),
        ('HS', Float64),
        ('GBIL', Float64),
        ('TP', Float64),
        ('ALB', Float64),
        ('AMY', Int64),
        ('PAMY', Int64),
        ('LIP', Int64),
        ('CHE', Float64),
        ('AP', Int64),
        ('ASAT', Int64),
        ('ALAT', Int64),
        ('GGT', Int64),
        ('LDH', Int64),
        ('CK', Int64),
        ('

In [10]:
df.is_duplicated().count

<bound method Series.count of shape: (11_753,)
Series: '' [bool]
[
	false
	false
	false
	false
	false
	…
	false
	false
	false
	false
	false
]>